# BFS & DFS — Test & Benchmark Notebook

Exercises `Graph.bfs()` and `Graph.dfs()` plus the general-purpose `Graph` construction
API they depend on (`add_edge`, `add_edges`, `add_adjacency_matrix`, `neighbors`, `edges`).
Roughly four kinds of cells appear, in this order:

1. **Construction & inspection** (cells 1–12) — build small graphs and read them back.
2. **Stress / perf smoke tests** (cells 13–18) — large random graphs, checking nothing
   crashes or leaks under load (no correctness assertions here, just "did it survive").
3. **Correctness tests** (cells 19–29, 35–43) — small hand-built graphs with an obvious
   expected answer, plus randomized cross-validation against `networkx` as an oracle.
4. **Benchmarks** (cells 30–34, 44–46) — `algokit` vs `networkx` timing on the same
   100k-vertex / 300k-edge random graph.

**Reproducibility note:** the benchmark cells reuse the module-level names `G_algo` /
`G_nx`. The BFS benchmark (cells 30–34) and the DFS benchmark (cells 44–46) each build
*their own* graph under those same names — run each block **top to bottom as a group**;
don't jump straight to cell 46 expecting the BFS-benchmark graph still to be there.


## Setup

Clones the repository and installs it in editable mode, then confirms the import works
and prints the package version — identical setup to the other test notebooks in this
project.

In [ ]:
!git clone https://github.com/ayush09062004/algokit-graph.git
%cd algokit-graph

Cloning into 'algokit-graph'...
remote: Enumerating objects: 829, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 829 (delta 5), reused 8 (delta 3), pack-reused 814 (from 1)
Receiving objects: 100% (829/829), 52.71 MiB | 10.07 MiB/s, done.
Resolving deltas: 100% (427/427), done.
/content/algokit-graph


In [ ]:
!pip install -e .

Obtaining file:///content/algokit-graph
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for algokit (pyproject.toml) ... done
  Created wheel for algokit: filename=algokit-0.1.0-cp312-cp312-linux_x86_64.whl size=167247 sha256=65aded4323476830785d995d6a194ef888efa6b9f86c65e403108535413588ea
  Stored in directory: /tmp/pip-ephem-wheel-cache-207_m6ye/wheels/6f/cc/a1/7c6a46c6b115b53184cea0dc45b2d81e45921650c6221c31c8
Successfully built algokit


In [ ]:
import algokit

print("Version:", algokit.__version__)
print(algokit.__doc__)

Version: 1.0.0
AlgoKit Python Bindings


## Construction & inspection

Creates an empty undirected graph with 5 vertices and no edges yet, and checks the basic
metadata accessors (`__repr__`, `vertex_count`, `edge_count`, `is_directed`).

**Expect:** `vertex_count() == 5`, `edge_count() == 0`, `is_directed() == False`.

In [ ]:
g = algokit.Graph.undirected(5)

print(g)
print(g.vertex_count())
print(g.edge_count())
print(g.is_directed())

Graph(num_vertices=5, num_edges=0, directed=False)
5
0
False


Same check for a fresh **directed** graph with 4 vertices.

**Expect:** `is_directed() == True`, `edge_count() == 0`.

In [ ]:
dg = algokit.Graph.directed(4)

print(dg)
print(dg.vertex_count())
print(dg.edge_count())
print(dg.is_directed())

Graph(num_vertices=4, num_edges=0, directed=True)
4
0
True


Adds three edges to the undirected graph from above: two with an implicit
weight of `1.0`, and one explicit weight of `5`. Reprinting `g` shows the edge count go
from 0 to 3.

In [ ]:
g.add_edge(0,1)
g.add_edge(0,2,5)
g.add_edge(1,3)

print(g)

Graph(num_vertices=5, num_edges=3, directed=False)


Reads back the adjacency list with `neighbors()` for three different vertices:
one with two edges, one with one edge, and one (`4`) with none at all — confirming
`neighbors()` on a degree-0 vertex returns an empty list rather than erroring.

In [ ]:
print(g.neighbors(0))
print(g.neighbors(1))
print(g.neighbors(4))

[Edge(to_vertex=1, weight=1), Edge(to_vertex=2, weight=5)]
[Edge(to_vertex=0, weight=1), Edge(to_vertex=3, weight=1)]
[]


`edges()` returns the full edge list as `GraphEdge` objects (each carrying an
explicit `from_vertex`), as opposed to `neighbors()` which returns `Edge` objects scoped
to a single vertex.

In [ ]:
print(g.edges())

[GraphEdge(from_vertex=0, to_vertex=1, weight=1), GraphEdge(from_vertex=0, to_vertex=2, weight=5), GraphEdge(from_vertex=1, to_vertex=3, weight=1)]


Inspects a single `Edge` object pulled from `neighbors(0)` — its `__repr__`,
and its two fields, `to_vertex` and `weight`. Note there's no `from_vertex` here; it's
implicit (you already know it because you asked for vertex `0`'s neighbors).

In [ ]:
e = g.neighbors(0)[0]

print(e)
print(e.to_vertex)
print(e.weight)

Edge(to_vertex=1, weight=1)
1
1.0


Same inspection for a `GraphEdge` object pulled from `edges()` — this one does
carry `from_vertex` explicitly, alongside `to_vertex` and `weight`.

In [ ]:
ge = g.edges()[0]

print(ge)
print(ge.from_vertex)
print(ge.to_vertex)
print(ge.weight)

GraphEdge(from_vertex=0, to_vertex=1, weight=1)
0
1
1.0


### Bulk construction with `add_edges()`

Builds a 6-vertex undirected graph from a list of tuples in one call, mixing 2-tuples
(default weight) and a 3-tuple (explicit weight `2.5`).

In [ ]:
g2 = algokit.Graph.undirected(6)

g2.add_edges([
    (0,1),
    (1,2),
    (2,3),
    (3,4,2.5),
    (4,5)
])

print(g2)
print(g2.edges())

Graph(num_vertices=6, num_edges=5, directed=False)
[GraphEdge(from_vertex=0, to_vertex=1, weight=1), GraphEdge(from_vertex=1, to_vertex=2, weight=1), GraphEdge(from_vertex=2, to_vertex=3, weight=1), GraphEdge(from_vertex=3, to_vertex=4, weight=2.5), GraphEdge(from_vertex=4, to_vertex=5, weight=1)]


### Construction from an adjacency matrix

Builds a 4-vertex undirected graph from a symmetric 0/1 matrix via
`add_adjacency_matrix()`. Only the upper triangle is read for undirected graphs, so each
edge is added once even though the matrix has both `matrix[i][j]` and `matrix[j][i]` set.

In [ ]:
g3 = algokit.Graph.undirected(4)

g3.add_adjacency_matrix([
    [0,1,0,0],
    [1,0,1,1],
    [0,1,0,0],
    [0,1,0,0]
])

print(g3)
print(g3.edges())

Graph(num_vertices=4, num_edges=3, directed=False)
[GraphEdge(from_vertex=0, to_vertex=1, weight=1), GraphEdge(from_vertex=1, to_vertex=2, weight=1), GraphEdge(from_vertex=1, to_vertex=3, weight=1)]


### Error handling — invalid vertex ids

Runs four calls that should all fail: `add_edge`/`neighbors` with a negative id and with
an id past `vertex_count()`. Confirms each one raises rather than silently doing the
wrong thing (e.g. wrapping around, or writing out of bounds).

In [ ]:
tests = [
    lambda: g.add_edge(-1, 0),
    lambda: g.add_edge(10, 0),
    lambda: g.neighbors(10),
    lambda: g.neighbors(-1),
]

for i, t in enumerate(tests, 1):
    try:
        t()
        print(f"❌ Test {i}: No exception")
    except Exception as e:
        print(f"✅ Test {i}: {type(e).__name__}: {e}")

✅ Test 1: TypeError: add_edge(): incompatible function arguments. The following argument types are supported:
    1. (self: algokit.Graph, from_vertex: typing.SupportsInt | typing.SupportsIndex, to_vertex: typing.SupportsInt | typing.SupportsIndex, weight: typing.SupportsFloat | typing.SupportsIndex = 1.0) -> None

Invoked with: Graph(num_vertices=5, num_edges=3, directed=False), -1, 0
✅ Test 2: IndexError: Vertex index is out of range.
✅ Test 3: IndexError: Vertex index is out of range.
✅ Test 4: TypeError: neighbors(): incompatible function arguments. The following argument types are supported:
    1. (self: algokit.Graph, vertex: typing.SupportsInt | typing.SupportsIndex) -> list[algokit.Edge]

Invoked with: Graph(num_vertices=5, num_edges=3, directed=False), -1


## Stress / perf smoke tests

These cells intentionally don't assert anything — they just push the C++ allocator hard
and confirm the process survives (no crash, no runaway memory) under `gc.collect()`.
Useful as a quick regression check after touching memory-management code.

Constructs 10,000 empty 1000-vertex undirected graphs back to back and discards each one.

In [ ]:
import gc
import time

start = time.time()

for _ in range(10000):
    g = algokit.Graph.undirected(1000)

del g
gc.collect()

print(f"Done in {time.time()-start:.2f} seconds")

Done in 0.12 seconds


Same idea, but each of the 1000 graphs also gets 999 edges added (a full
Hamiltonian chain) before being discarded.

In [ ]:
import gc
import time

start = time.time()

for _ in range(1000):
    g = algokit.Graph.undirected(1000)

    for i in range(999):
        g.add_edge(i, i + 1)

del g
gc.collect()

print(f"Done in {time.time()-start:.2f} seconds")

Done in 0.79 seconds


500 graphs, each with 5000 random edges added — exercises the general
random-insertion path rather than a predictable chain pattern.

In [ ]:
import random
import gc

for _ in range(500):
    g = algokit.Graph.undirected(1000)

    for _ in range(5000):
        u = random.randint(0, 999)
        v = random.randint(0, 999)
        g.add_edge(u, v)

del g
gc.collect()

print("Random graph stress test passed!")

Random graph stress test passed!


Alternates between directed and undirected graphs across 1000 iterations
(500 of each), each with 2000 random edges — checks that switching graph "kind" doesn't
leak state between constructions.

In [ ]:
import random
import gc

for i in range(1000):

    if i % 2:
        g = algokit.Graph.directed(500)
    else:
        g = algokit.Graph.undirected(500)

    for _ in range(2000):
        u = random.randint(0,499)
        v = random.randint(0,499)
        g.add_edge(u,v)

del g
gc.collect()

print("Mixed graph stress test passed!")

Mixed graph stress test passed!


Hits several read APIs (`vertex_count`, `edge_count`, `edges`, `neighbors`)
repeatedly across 500 freshly built random graphs, not just the constructor/insert path
exercised above.

In [ ]:
import random
import gc

for _ in range(500):

    g = algokit.Graph.undirected(300)

    for _ in range(1500):
        u = random.randint(0,299)
        v = random.randint(0,299)
        g.add_edge(u,v)

    g.vertex_count()
    g.edge_count()
    g.edges()

    for v in range(50):
        g.neighbors(v)

del g
gc.collect()

print("API stress test passed!")

API stress test passed!


A single large graph: 100,000 vertices with 1,000,000 `add_edge()` calls (one
edge at a time, not bulk), timed end to end. This is the "worst case" insertion pattern —
see the `ToyProject1` notebook for a bulk-vs-single-edge speed comparison.

In [ ]:
import random
import time

start = time.time()

g = algokit.Graph.undirected(100000)

for _ in range(1000000):
    u = random.randint(0,99999)
    v = random.randint(0,99999)
    g.add_edge(u,v)

print(g)

print(f"Time: {time.time()-start:.2f}s")

Graph(num_vertices=100000, num_edges=1000000, directed=False)
Time: 2.67s


## BFS correctness — hand-built graphs

A 6-vertex undirected graph shaped like a small tree with one extra branch:
`0-1, 0-2, 1-3, 2-4, 4-5`. Running `bfs(0)` and printing all three result accessors
(`order`, `distance`, `parent`) together, next to the graph, makes it easy to check the
BFS tree by eye.

**Expect:** `order = [0, 1, 2, 3, 4, 5]`, `distance = [0, 1, 1, 2, 2, 3]`.

In [ ]:
import algokit

g = algokit.Graph.undirected(6)

g.add_edges([
    (0,1),
    (0,2),
    (1,3),
    (2,4),
    (4,5)
])

print(g)

bfs = g.bfs(0)

print(bfs)
print("Order:", bfs.order())
print("Distance:", bfs.distance())
print("Parent:", bfs.parent())

Graph(num_vertices=6, num_edges=5, directed=False)
BFSResult(order=[0, 1, 2, 3, 4, ...])
Order: [0, 1, 2, 3, 4, 5]
Distance: [0, 1, 1, 2, 2, 3]
Parent: [-1, 0, 0, 1, 2, 4]


Reruns BFS on the *same* graph but from vertex `5` instead of `0` — a leaf
at the far end of the tree — to check the traversal direction-agnostic property of BFS
on an undirected graph.

In [ ]:
bfs = g.bfs(5)

print("Order:", bfs.order())
print("Distance:", bfs.distance())
print("Parent:", bfs.parent())

Order: [5, 4, 2, 0, 1, 3]
Distance: [3, 4, 2, 5, 1, 0]
Parent: [2, 0, 4, 1, 5, -1]


### Edge case — single-vertex graph

A graph with exactly one vertex and no edges. `bfs(0)` should still succeed trivially.

**Expect:** `order = [0]`, `distance = [0]`, `parent = [-1]`.

In [ ]:
g = algokit.Graph.undirected(1)

bfs = g.bfs(0)

print(bfs.order())
print(bfs.distance())
print(bfs.parent())

[0]
[0]
[-1]


### Edge case — disconnected graph

6 vertices, but only edges `0-1, 1-2` and a separate `3-4` component; vertex `5` is
completely isolated. BFS from `0` should only reach `{0,1,2}`.

**Expect:** `order = [0, 1, 2]`; vertices `3`, `4`, `5` don't appear in `order` at all,
and their `distance`/`parent` entries stay at the "unreached" sentinel.

In [ ]:
g = algokit.Graph.undirected(6)

g.add_edges([
    (0,1),
    (1,2),
    (3,4)
])

bfs = g.bfs(0)

print(bfs.order())
print(bfs.distance())
print(bfs.parent())

[0, 1, 2]
[0, 1, 2, -1, -1, -1]
[-1, 0, 1, -1, -1, -1]


### Edge case — source has exactly one neighbor

5 vertices, single edge `0-1`. BFS from `4` (an isolated vertex) should reach only
itself.

**Expect:** `order = [4]`.

In [ ]:
g = algokit.Graph.undirected(5)

g.add_edge(0,1)

bfs = g.bfs(4)

print(bfs.order())
print(bfs.distance())
print(bfs.parent())

[4]
[-1, -1, -1, -1, 0]
[-1, -1, -1, -1, -1]


### BFS on a graph with a cycle

A 4-cycle `0-1-2-3-0`. Unlike DFS, BFS doesn't need special cycle handling — the
`distance`/`visited` check naturally prevents revisiting — but it's worth confirming the
parent pointers still describe a valid tree (no cycles in the *BFS tree* itself, even
though the underlying graph has one).

In [ ]:
g = algokit.Graph.undirected(4)

g.add_edges([
    (0,1),
    (1,2),
    (2,3),
    (3,0)
])

bfs = g.bfs(0)

print(bfs.order())
print(bfs.distance())
print(bfs.parent())

[0, 1, 3, 2]
[0, 1, 2, 1]
[-1, 0, 1, 0]


### BFS with a self-loop

3 vertices; `0` has a self-loop (`0-0`) in addition to normal edges `0-1, 1-2`. Confirms
a self-loop doesn't confuse the "already visited" check or cause an infinite loop.

In [ ]:
g = algokit.Graph.undirected(3)

g.add_edge(0,0)
g.add_edge(0,1)
g.add_edge(1,2)

bfs = g.bfs(0)

print(bfs.order())

[0, 1, 2]


### Error handling — out-of-range source

`bfs(100)` on a graph that doesn't have 100 vertices should raise immediately rather than
running off the end of an internal array.

**Expect:** an `IndexError` / `std::out_of_range`.

In [ ]:
try:
    g.bfs(100)
except Exception as e:
    print(type(e).__name__, e)

IndexError Vertex index is out of range.


## Cross-validation against `networkx`

Installs `networkx`, used purely as an independent reference implementation to validate
`algokit`'s BFS/DFS output on graphs too large or numerous to check by hand.

In [ ]:
!pip install -q networkx

Generates 200 random undirected graphs (2–50 vertices, random edge count),
builds the *same* graph in both `networkx` and `algokit`, runs BFS from a random source
in each, and asserts the per-vertex distance arrays match exactly (`-1` used as the
networkx-side "unreached" sentinel to match `algokit`'s convention).

**Expect:** `"✅ All 200 randomized BFS tests passed!"` — if a mismatch is ever found, the
loop prints the offending source, both distance arrays, and stops immediately so the
failing case can be inspected.

In [ ]:
import random
import networkx as nx
import algokit

NUM_TESTS = 200

for test in range(NUM_TESTS):

    n = random.randint(2, 50)
    m = random.randint(0, n * (n - 1) // 2)

    G_nx = nx.Graph()
    G_nx.add_nodes_from(range(n))

    G_algo = algokit.Graph.undirected(n)

    edges = set()

    while len(edges) < m:
        u = random.randrange(n)
        v = random.randrange(n)

        if u == v:
            continue

        e = tuple(sorted((u, v)))

        if e not in edges:
            edges.add(e)
            G_nx.add_edge(u, v)
            G_algo.add_edge(u, v)

    source = random.randrange(n)

    bfs = G_algo.bfs(source)

    # AlgoKit distances
    dist_algo = bfs.distance()

    # NetworkX distances
    dist_nx = [-1] * n
    nx_lengths = nx.single_source_shortest_path_length(G_nx, source)

    for v, d in nx_lengths.items():
        dist_nx[v] = d

    if dist_algo != dist_nx:
        print(f"❌ Test {test} failed")
        print("Source:", source)
        print("AlgoKit :", dist_algo)
        print("NetworkX:", dist_nx)
        break

else:
    print(f"✅ All {NUM_TESTS} randomized BFS tests passed!")

✅ All 200 randomized BFS tests passed!


Same randomized BFS-vs-networkx check as above, just with 1000 iterations
instead of 200 for a higher-confidence run (slower, so it's kept as a separate cell you
can skip if you already trust the smaller sample).

In [ ]:
import random
import networkx as nx
import algokit

NUM_TESTS = 1000

for test in range(NUM_TESTS):

    n = random.randint(2, 50)
    m = random.randint(0, n * (n - 1) // 2)

    G_nx = nx.Graph()
    G_nx.add_nodes_from(range(n))

    G_algo = algokit.Graph.undirected(n)

    edges = set()

    while len(edges) < m:
        u = random.randrange(n)
        v = random.randrange(n)

        if u == v:
            continue

        e = tuple(sorted((u, v)))

        if e not in edges:
            edges.add(e)
            G_nx.add_edge(u, v)
            G_algo.add_edge(u, v)

    source = random.randrange(n)

    bfs = G_algo.bfs(source)

    # AlgoKit distances
    dist_algo = bfs.distance()

    # NetworkX distances
    dist_nx = [-1] * n
    nx_lengths = nx.single_source_shortest_path_length(G_nx, source)

    for v, d in nx_lengths.items():
        dist_nx[v] = d

    if dist_algo != dist_nx:
        print(f"❌ Test {test} failed")
        print("Source:", source)
        print("AlgoKit :", dist_algo)
        print("NetworkX:", dist_nx)
        break

else:
    print(f"✅ All {NUM_TESTS} randomized BFS tests passed!")

✅ All 1000 randomized BFS tests passed!


## BFS benchmark: `algokit` vs `networkx`

Builds one large shared random graph — 100,000 vertices, 300,000 edges — in both
`networkx` (`G_nx`) and `algokit` (`G_algo`), used by all the following benchmark cells.
Building the edge set once here (instead of per-run) keeps the timing below focused on
the traversal itself, not graph construction.

In [ ]:
import random
import networkx as nx
import algokit

n = 100000
m = 300000

G_nx = nx.Graph()
G_nx.add_nodes_from(range(n))

G_algo = algokit.Graph.undirected(n)

edges = set()

while len(edges) < m:
    u = random.randrange(n)
    v = random.randrange(n)

    if u == v:
        continue

    e = tuple(sorted((u, v)))

    if e not in edges:
        edges.add(e)
        G_nx.add_edge(u, v)
        G_algo.add_edge(u, v)

Runs each implementation's BFS once, untimed — just to warm up and confirm
both produce output before the timed runs below.

In [ ]:
G_algo.bfs(0)
nx.single_source_shortest_path_length(G_nx, 0)

{0: 0,
 65986: 1,
 30723: 1,
 41673: 1,
 60740: 1,
 2629: 1,
 61435: 1,
 21466: 1,
 31379: 2,
 74991: 2,
 37668: 2,
 28426: 2,
 79817: 2,
 87056: 2,
 51161: 2,
 20021: 2,
 86003: 2,
 35148: 2,
 59032: 2,
 88930: 2,
 58550: 2,
 32531: 2,
 95958: 2,
 21108: 2,
 54821: 2,
 90056: 2,
 38: 2,
 67295: 2,
 68113: 2,
 37351: 2,
 78271: 2,
 63920: 2,
 98629: 2,
 4905: 2,
 25477: 2,
 74420: 2,
 90369: 2,
 12973: 2,
 9285: 2,
 65507: 2,
 56879: 2,
 2904: 2,
 96792: 2,
 80863: 2,
 53565: 2,
 57429: 2,
 12828: 2,
 33621: 2,
 91079: 2,
 61850: 2,
 74635: 2,
 26912: 2,
 32645: 2,
 86081: 2,
 10038: 2,
 68308: 3,
 93388: 3,
 93920: 3,
 44561: 3,
 45333: 3,
 43488: 3,
 62437: 3,
 47792: 3,
 24972: 3,
 36407: 3,
 79337: 3,
 37039: 3,
 21778: 3,
 82298: 3,
 58260: 3,
 39222: 3,
 77636: 3,
 73379: 3,
 15623: 3,
 40619: 3,
 87618: 3,
 44567: 3,
 35482: 3,
 17950: 3,
 67612: 3,
 35099: 3,
 34607: 3,
 61246: 3,
 46220: 3,
 4886: 3,
 81489: 3,
 45543: 3,
 41821: 3,
 73279: 3,
 53795: 3,
 86657: 3,
 70076: 3,


Times `algokit`'s `bfs(0)` over 20 runs on the shared graph and records the
average per-run time in `algo_time`.

In [ ]:
import time

RUNS = 20

start = time.perf_counter()

for _ in range(RUNS):
    G_algo.bfs(0)

algo_time = (time.perf_counter() - start) / RUNS

Times `networkx`'s `single_source_shortest_path_length` (its BFS-distance
equivalent) over the same 20 runs for a fair comparison, recorded as `nx_time`.

In [ ]:
start = time.perf_counter()

for _ in range(RUNS):
    nx.single_source_shortest_path_length(G_nx, 0)

nx_time = (time.perf_counter() - start) / RUNS

Prints both average times side by side and the resulting speedup factor.

In [ ]:
print(f"AlgoKit : {algo_time:.6f} s")
print(f"NetworkX: {nx_time:.6f} s")
print(f"Speedup : {nx_time/algo_time:.2f}x")

AlgoKit : 0.031444 s
NetworkX: 0.493239 s
Speedup : 15.69x


## DFS correctness — hand-built graphs

Same 6-vertex tree-shaped graph used for the BFS examples above. Running `dfs(0)` and
printing `order`, `parent`, `discovery_time`, and `finish_time` together lets you check
the DFS tree and timestamps by eye.

In [ ]:
import algokit

g = algokit.Graph.undirected(6)

g.add_edges([
    (0,1),
    (0,2),
    (1,3),
    (2,4),
    (4,5)
])

dfs = g.dfs(0)

print(dfs)
print("Order      :", dfs.order())
print("Parent     :", dfs.parent())
print("Discovery  :", dfs.discovery_time())
print("Finish     :", dfs.finish_time())

DFSResult(order=[0, 1, 3, 2, 4, ...])
Order      : [0, 1, 3, 2, 4, 5]
Parent     : [-1, 0, 0, 1, 2, 4]
Discovery  : [0, 1, 5, 2, 6, 7]
Finish     : [11, 4, 10, 3, 9, 8]


Reruns DFS on the same graph from leaf vertex `5` instead of `0`.

In [ ]:
dfs = g.dfs(5)

print("Order      :", dfs.order())
print("Parent     :", dfs.parent())
print("Discovery  :", dfs.discovery_time())
print("Finish     :", dfs.finish_time())

Order      : [5, 4, 2, 0, 1, 3]
Parent     : [2, 0, 4, 1, 5, -1]
Discovery  : [3, 4, 2, 5, 1, 0]
Finish     : [8, 7, 9, 6, 10, 11]


### Edge case — single-vertex graph

`dfs(0)` on a graph with just one vertex, mirroring the BFS single-vertex check above.

In [ ]:
g = algokit.Graph.undirected(1)

dfs = g.dfs(0)

print(dfs.order())
print(dfs.parent())
print(dfs.discovery_time())
print(dfs.finish_time())

[0]
[-1]
[0]
[1]


### Edge case — disconnected graph

Same disconnected 6-vertex graph as the BFS disconnected test — DFS from `0` should only
visit `{0, 1, 2}`.

In [ ]:
g = algokit.Graph.undirected(6)

g.add_edges([
    (0,1),
    (1,2),
    (3,4)
])

dfs = g.dfs(0)

print(dfs.order())
print(dfs.parent())
print(dfs.discovery_time())
print(dfs.finish_time())

[0, 1, 2]
[-1, 0, 1, -1, -1, -1]
[0, 1, 2, -1, -1, -1]
[5, 4, 3, -1, -1, -1]


### DFS on a graph with a cycle

The same 4-cycle used in the BFS cycle test, to confirm DFS's `visited` tracking handles
a cycle without infinite-looping.

In [ ]:
g = algokit.Graph.undirected(4)

g.add_edges([
    (0,1),
    (1,2),
    (2,3),
    (3,0)
])

dfs = g.dfs(0)

print(dfs.order())
print(dfs.parent())

[0, 1, 2, 3]
[-1, 0, 1, 2]


### DFS with a self-loop

Same self-loop setup as the BFS test — `0` has a self-loop plus edges `0-1, 1-2`.

In [ ]:
g = algokit.Graph.undirected(3)

g.add_edge(0,0)
g.add_edge(0,1)
g.add_edge(1,2)

dfs = g.dfs(0)

print(dfs.order())

[0, 1, 2]


### Error handling — out-of-range source

`dfs(100)` should raise the same way `bfs(100)` did.

In [ ]:
try:
    g.dfs(100)
except Exception as e:
    print(type(e).__name__, e)

IndexError Vertex index is out of range.


Pretty-prints discovery/finish timestamps per vertex, in visit order — a more
readable view of the same `discovery_time()`/`finish_time()` arrays than the raw lists
printed in the cells above.

In [ ]:
dfs = g.dfs(0)

disc = dfs.discovery_time()
finish = dfs.finish_time()

for v in dfs.order():
    print(
        f"Vertex {v}: "
        f"discover={disc[v]}, "
        f"finish={finish[v]}"
    )

Vertex 0: discover=0, finish=5
Vertex 1: discover=1, finish=4
Vertex 2: discover=2, finish=3


Randomized DFS-vs-networkx cross-validation: 500 random graphs, comparing the
*set* of visited vertices from `algokit`'s `dfs()` against `networkx.dfs_preorder_nodes`.
(Sets are compared rather than exact order, since DFS visit order is only required to be
*a* valid DFS order, not necessarily identical between two different implementations.)

**Expect:** `"✅ All 500 randomized DFS tests passed!"`

In [ ]:
import random
import networkx as nx
import algokit

NUM_TESTS = 500

for test in range(NUM_TESTS):

    n = random.randint(2, 50)
    m = random.randint(0, n*(n-1)//2)

    G_nx = nx.Graph()
    G_nx.add_nodes_from(range(n))

    G_algo = algokit.Graph.undirected(n)

    edges = set()

    while len(edges) < m:
        u = random.randrange(n)
        v = random.randrange(n)

        if u == v:
            continue

        e = tuple(sorted((u, v)))

        if e not in edges:
            edges.add(e)
            G_nx.add_edge(u, v)
            G_algo.add_edge(u, v)

    source = random.randrange(n)

    dfs = G_algo.dfs(source)

    visited_algo = set(dfs.order())
    visited_nx = set(nx.dfs_preorder_nodes(G_nx, source))

    if visited_algo != visited_nx:
        print("❌ Failed")
        print(test)
        print(visited_algo)
        print(visited_nx)
        break

else:
    print(f"✅ All {NUM_TESTS} randomized DFS tests passed!")

✅ All 500 randomized DFS tests passed!


## DFS benchmark: `algokit` vs `networkx`

Builds a fresh 100,000-vertex / 300,000-edge random graph for the DFS benchmark — this
**reassigns** `G_algo`/`G_nx`, replacing the graph built earlier for the BFS benchmark.

In [ ]:
import random
import networkx as nx
import algokit

n = 100000
m = 300000

G_nx = nx.Graph()
G_nx.add_nodes_from(range(n))

G_algo = algokit.Graph.undirected(n)

edges = set()

while len(edges) < m:
    u = random.randrange(n)
    v = random.randrange(n)

    if u == v:
        continue

    e = tuple(sorted((u, v)))

    if e not in edges:
        edges.add(e)
        G_nx.add_edge(u, v)
        G_algo.add_edge(u, v)

Runs each implementation's DFS once, untimed, as a warm-up/sanity check
before timing.

In [ ]:
G_algo.dfs(0)
list(nx.dfs_preorder_nodes(G_nx, source=0))

[0,
 17202,
 54569,
 22317,
 52108,
 52226,
 16246,
 2257,
 66378,
 14031,
 83439,
 73772,
 57770,
 63049,
 82535,
 11273,
 98006,
 51091,
 96220,
 25840,
 45987,
 52428,
 73265,
 80478,
 89157,
 34045,
 21926,
 49888,
 84165,
 40183,
 47302,
 43438,
 3086,
 52530,
 69248,
 51464,
 18185,
 64584,
 76914,
 18556,
 77363,
 70804,
 40139,
 37136,
 88688,
 58784,
 78731,
 87530,
 4294,
 85431,
 97948,
 60956,
 48742,
 47780,
 69603,
 46115,
 92329,
 17904,
 41905,
 97256,
 89910,
 21616,
 45728,
 31515,
 78497,
 15613,
 66153,
 88371,
 84250,
 54951,
 8230,
 4023,
 81427,
 13707,
 66092,
 32658,
 11891,
 32101,
 62456,
 15935,
 82979,
 15149,
 80831,
 43865,
 13403,
 63688,
 70686,
 81526,
 27665,
 27154,
 85342,
 47193,
 85622,
 55315,
 41081,
 39198,
 82712,
 5947,
 73557,
 84957,
 4357,
 23768,
 91414,
 61367,
 82040,
 60573,
 46828,
 75473,
 45719,
 62369,
 62876,
 70663,
 86126,
 63602,
 7334,
 78867,
 52231,
 61430,
 99194,
 64471,
 71740,
 50974,
 36937,
 20379,
 87926,
 32501,
 406

Times both `algokit`'s `dfs(0)` and `networkx`'s `dfs_preorder_nodes` over 20
runs each on the shared graph, then prints the average times and speedup — combined into
one cell this time (contrast with the BFS benchmark above, which split timing across
separate cells).

In [ ]:
import time

RUNS = 20

start = time.perf_counter()

for _ in range(RUNS):
    G_algo.dfs(0)

algo_time = (time.perf_counter() - start) / RUNS

start = time.perf_counter()

for _ in range(RUNS):
    list(nx.dfs_preorder_nodes(G_nx, source=0))

nx_time = (time.perf_counter() - start) / RUNS

print(f"AlgoKit : {algo_time:.6f} s")
print(f"NetworkX: {nx_time:.6f} s")
print(f"Speedup : {nx_time / algo_time:.2f}x")

AlgoKit : 0.066790 s
NetworkX: 1.022083 s
Speedup : 15.30x


## Aside

Not a BFS/DFS test — a stray exploratory call to `help()` on
`ConnectedComponentsResult`'s docstring, likely left over from cross-referencing another
notebook. Harmless to run; just prints the class's built-in help text.

In [ ]:
help(algokit.ConnectedComponentsResult)

Help on class ConnectedComponentsResult in module algokit:

class ConnectedComponentsResult(pybind11_builtins.pybind11_object)
 |  Result of connected components computation.
 |
 |  Attributes:
 |      component_count (int): Number of connected components.
 |      component_id (list[int]): Component ID for each vertex.
 |      components (list[list[int]]): List of vertices in each component.
 |
 |  Method resolution order:
 |      ConnectedComponentsResult
 |      pybind11_builtins.pybind11_object
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __init__(self, /, *args, **kwargs)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |
 |  __repr__(...)
 |      __repr__(self: algokit.ConnectedComponentsResult) -> str
 |
 |      Returns a concise representation of the connected components result.
 |
 |  component_count(...)
 |      component_count(self: algokit.ConnectedComponentsResult) -> int
 |
 |      Returns the number of connected components.
 |
 |  com